# 13 – Khoảng tin cậy ghép cặp theo người nói (bổ sung theo phản biện)

Không train, không trích đặc trưng. Notebook dùng lại:
- đặc trưng đã làm nhiễu của notebook 12 (`results/robustness/feats_*`) và head của mô hình kết hợp (notebook 10);
- dự đoán của các baseline (notebook 02).

Tính:
1. **CI ghép cặp theo người nói** cho chênh lệch accuracy giữa các điều kiện LTAS (own − norm, own − swap, …).
2. **CI theo người nói cho Bảng II** (baseline), để mọi bảng có cùng một cách bootstrap.

- **Runtime:** CPU là đủ, khoảng 2–3 phút.
- **Kết quả:** `results/audit/paired_ci.json`. Gửi file này cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
N_BOOT = 2000
CONDS = ["ltas_own", "ltas_norm", "ltas_swap", "telephone", "noise20"]
BASELINES = ["acoustic_ltas", "acoustic_mfcc", "text_tfidf", "text_phobert", "xlsr_best_layer_linear", "xlsr_wsum_linear", "xlsr_wsum_mlp"]

In [ ]:
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import f1_score
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
RES_DIR = f"{DRIVE_ROOT}/results"; FIN = f"{RES_DIR}/final"; ROB = f"{RES_DIR}/robustness"; OUT = f"{RES_DIR}/audit"
os.makedirs(OUT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
te = (meta["split"].values == "test") & use
yt = y[te]
_, spk_id = np.unique(meta.loc[te, "spk"].astype(str).values, return_inverse=True); K = spk_id.max() + 1
B_SPK = np.random.default_rng(0).integers(0, K, (N_BOOT, K)); spk_n = np.bincount(spk_id, minlength=K).astype(float)

def ci_spk(v):
    s = np.bincount(spk_id, weights=v, minlength=K)
    return [round(float(x), 4) for x in np.percentile(s[B_SPK].sum(1) / spk_n[B_SPK].sum(1), [2.5, 97.5])]
print(f"test: {len(yt)} câu, {K} người nói")
REPORT = {}

## 1. CI ghép cặp giữa các điều kiện LTAS

In [ ]:
class HeadD(nn.Module):   # giống notebook 10/12
    def __init__(s, LB, DB, LC, DC):
        super().__init__()
        s.register_buffer("mu_B", torch.zeros(LB, DB)); s.register_buffer("sd_B", torch.ones(LB, DB))
        s.register_buffer("mu_C", torch.zeros(LC, DC)); s.register_buffer("sd_C", torch.ones(LC, DC))
        s.w = nn.ParameterDict({"B": nn.Parameter(torch.zeros(LB)), "C": nn.Parameter(torch.zeros(LC))})
        s.mlp = nn.Sequential(nn.Linear(DB + DC, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, xb, xc):
        parts = []
        for k, x in (("B", xb), ("C", xc)):
            z = (x.float() - getattr(s, f"mu_{k}")) / getattr(s, f"sd_{k}")
            parts.append((z * s.w[k].softmax(0)[None, :, None]).sum(1))
        return s.mlp(torch.cat(parts, -1))

PRED = {}
for l in ("ls", "geols"):
    PRED[("clean", l)] = np.load(f"{FIN}/test_probs_D_fusion_{l}.npy").argmax(1)
avail = [c for c in CONDS if os.path.exists(f"{ROB}/feats_{c}/B.npy") and os.path.exists(f"{ROB}/feats_{c}/C.npy")]
print("Điều kiện có cache:", avail)
for c in avail:
    xb, xc = np.load(f"{ROB}/feats_{c}/B.npy"), np.load(f"{ROB}/feats_{c}/C.npy")
    for l in ("ls", "geols"):
        ps = []
        for s in (0, 1, 2):
            h = HeadD(xb.shape[1], xb.shape[2], xc.shape[1], xc.shape[2]).to(device)
            h.load_state_dict(torch.load(f"{FIN}/head_D_{l}_seed{s}.pt", map_location=device)); h.eval()
            with torch.no_grad():
                ps.append(np.concatenate([h(torch.tensor(xb[i:i + 512], device=device), torch.tensor(xc[i:i + 512], device=device)).softmax(-1).cpu().numpy()
                                          for i in range(0, len(xb), 512)]))
        PRED[(c, l)] = np.mean(ps, 0).argmax(1)

conds = ["clean"] + avail
acc = {f"{c}|{l}": round(float((PRED[(c, l)] == yt).mean()), 4) for c in conds for l in ("ls", "geols")}
print("Accuracy (kiểm tra khớp với notebook 12):", acc)
PAIRS = [("ltas_own", "ltas_norm"), ("ltas_own", "ltas_swap"), ("ltas_swap", "ltas_norm"),
         ("ltas_own", "clean"), ("ltas_norm", "clean"), ("ltas_swap", "clean"), ("telephone", "clean"), ("noise20", "clean")]
PD = {}
for a, b in PAIRS:
    if a in conds and b in conds:
        for l in ("ls", "geols"):
            d = (PRED[(a, l)] == yt).astype(float) - (PRED[(b, l)] == yt)
            PD[f"{a} - {b}|{l}"] = dict(acc_diff=round(float(d.mean()), 4), ci_spk=ci_spk(d))
print(json.dumps(PD, indent=1))
REPORT.update(accuracy=acc, paired_diffs=PD)

## 2. Bảng II với CI theo người nói

In [ ]:
T2 = {}
for n in BASELINES:
    f = f"{RES_DIR}/pred_{n}.npy"
    if os.path.exists(f):
        p = np.load(f); ok = (p == yt).astype(float)
        T2[n] = dict(acc=round(float(ok.mean()), 4), acc_ci_spk=ci_spk(ok), macro_f1=round(float(f1_score(yt, p, average="macro")), 4))
print(pd.DataFrame(T2).T.to_string())
REPORT["table2_speaker_ci"] = T2
json.dump(REPORT, open(f"{OUT}/paired_ci.json", "w"), indent=1)
print(f"Đã lưu {OUT}/paired_ci.json – gửi file này cho Claude.")